<p style="font-size:300%; background-color:#ffe6f0; color:#0044cc; text-align:center; line-height:80px; margin:0; padding:0;">
  <b>Sistema Experto: Verificación de Piezas Mecánicas</b>
</p>

<p style="font-size:240%; background-color:#ffe6f0; color:#cc0000; text-align:center; line-height:60px; margin:0; padding:0;">
  <b>Técnicas de Inteligencia Artificial</b>
</p>

<p style="font-size:200%; text-align:center; line-height:40px; margin:0;">
  <br><b>Prof. Flavio Prieto</b>
</p>

<p style="font-size:160%; text-align:center; line-height:25px; margin:0;">
  email: <a href="mailto:faprietoo@unal.edu.co">faprietoo@unal.edu.co</a>
</p>

<p style="font-size:180%; text-align:center; line-height:30px; margin:0;">
  <br><b>DEPARTAMENTO DE INGENIERÍA MECÁNICA Y MECATRÓNICA</b>
</p>

<p style="font-size:180%; text-align:center; line-height:30px; margin:0;">
  Facultad de Ingeniería
</p>

<p style="font-size:160%; text-align:center; line-height:30px; margin:0;">
  Universidad Nacional de Colombia – Sede Bogotá
</p>

<br>

<p style="font-size:140%; text-align:center; line-height:30px; margin:0;">
  <b>Taller Integrado: Talleres 1 y 2 – Sistemas Expertos</b>
</p>

<p style="font-size:120%; text-align:center; line-height:25px; margin:0;">
  Chatbot experto por voz para verificar piezas mecánicas usando reglas,<br>
  factores de certeza MYCIN y módulo de explicación.
</p>


# 1. Objetivo

Desarrollar un **sistema experto integrado** que funcione como chatbot interactivo para la **verificación de calidad de piezas mecánicas mecanizadas** (ejes, bridas y tornillos).

El sistema debe:

1. Capturar respuestas del usuario mediante **reconocimiento de voz** (o entrada de texto alternativa).
2. Procesar la información con un **sistema experto basado en reglas** en lógica de primer orden.
3. Manejar **incertidumbre** con **factores de certeza (MYCIN)**.
4. Generar un **diagnóstico razonado** con módulo de explicación.
5. Presentar el resultado mediante **síntesis de voz**.

# 2. Descripción del Problema

En un proceso de manufactura se producen tres tipos de piezas mecanizadas:

- **Eje**
- **Brida**
- **Tornillo**

Cada pieza debe cumplir tolerancias dimensionales y de acabado. El sistema evalúa si una pieza debe **aceptarse**, **rechazarse** o enviarse a **revisión manual**, calculando un factor de certeza acumulado de rechazo.

### Tolerancias por tipo de pieza

| Tipo     | Diámetro (mm)  | Longitud (mm)   | Rugosidad máx. (Ra) | Material  |
|----------|----------------|------------------|----------------------|-----------|
| Eje      | 49.8 – 50.2    | 99.7 – 100.3     | 2.0                  | Acero     |
| Brida    | 80.0 – 80.5    | 19.5 – 20.5      | 2.0                  | Aluminio  |
| Tornillo | 9.8 – 10.2     | 19.0 – 21.0      | 1.5                  | Acero     |

### Factores de certeza por evidencia de fallo

| Evidencia de fallo               |   FC |
|----------------------------------|-----:|
| Diámetro fuera de tolerancia     | 0.40 |
| Longitud fuera de tolerancia     | 0.30 |
| Rugosidad excesiva               | 0.30 |
| Material incorrecto              | 0.60 |
| Grietas visibles                 | 0.70 |

# 3. Representación Formal del Conocimiento

Se usa una representación de hechos tipo **predicado** en lógica de primer orden.

### Hechos base (ejemplo para un eje)

```
pieza(eje)
diametro(eje_1, 50.4)
longitud(eje_1, 100.0)
rugosidad(eje_1, 1.8)
material(eje_1, acero)
grietas_visibles(eje_1, no)
deformacion_visible(eje_1, si)
uso_critico(eje_1, si)
```

### Predicados derivados

```
diametro_fuera_tolerancia(x)
longitud_fuera_tolerancia(x)
rugosidad_excesiva(x)
material_incorrecto(x)
grietas_detectadas(x)
deformacion_detectada(x)
dimension_fuera_especificacion(x)
integridad_comprometida(x)
pieza_rechazada(x)
pieza_aceptada(x)
requiere_revision(x)
```

# 4. Dependencias e Importaciones

In [ ]:
import re
import subprocess
import sys
from copy import deepcopy

# 5. Variables de Entrada

Se definen **8 variables de entrada** que el usuario debe proporcionar:

| #  | Variable              | Tipo    | Descripción                             |
|----|----------------------|---------|------------------------------------------|
| 1  | `tipo_pieza`         | texto   | eje, brida o tornillo                    |
| 2  | `diametro`           | numérico| Valor en mm                              |
| 3  | `longitud`           | numérico| Valor en mm                              |
| 4  | `rugosidad`          | numérico| Valor Ra                                 |
| 5  | `material`           | texto   | acero, aluminio u otro                   |
| 6  | `grietas_visibles`   | booleano| Sí / No                                  |
| 7  | `deformacion_visible`| booleano| Sí / No                                  |
| 8  | `uso_critico`        | booleano| Sí / No                                  |

Las primeras cinco variables provienen del enunciado del Taller 2. Las tres últimas amplían el dominio para obtener un diagnóstico más completo y no trivial.

# 6. Base de Hechos: Tolerancias y Especificaciones

Las evidencias de fallo se detectan como hechos binarios (presentes o no). El factor de certeza de cada evidencia está definido en las reglas correspondientes (R1–R6).

In [ ]:
TOLERANCIAS = {
    "eje": {
        "diametro_min": 49.8, "diametro_max": 50.2,
        "longitud_min": 99.7, "longitud_max": 100.3,
        "rugosidad_max": 2.0,
        "material_esperado": "acero"
    },
    "brida": {
        "diametro_min": 80.0, "diametro_max": 80.5,
        "longitud_min": 19.5, "longitud_max": 20.5,
        "rugosidad_max": 2.0,
        "material_esperado": "aluminio"
    },
    "tornillo": {
        "diametro_min": 9.8, "diametro_max": 10.2,
        "longitud_min": 19.0, "longitud_max": 21.0,
        "rugosidad_max": 1.5,
        "material_esperado": "acero"
    }
}

print("Tolerancias cargadas:")
for tipo, tol in TOLERANCIAS.items():
    print(f"  {tipo}: Ø {tol['diametro_min']}-{tol['diametro_max']} mm, "
          f"L {tol['longitud_min']}-{tol['longitud_max']} mm, "
          f"Ra ≤ {tol['rugosidad_max']}, material: {tol['material_esperado']}")

# 7. Base de Reglas

Se definen **12 reglas** con sus factores de certeza. Las reglas R1–R6 son reglas de detección directa de evidencias. Las reglas R7–R10 son **reglas encadenadas** que infieren conclusiones a partir de hechos derivados.

| ID  | Regla                                                                                 | FC   |
|-----|--------------------------------------------------------------------------------------|-----:|
| R1  | Si diámetro fuera de tolerancia → fallo de diámetro                                  | 0.40 |
| R2  | Si longitud fuera de tolerancia → fallo de longitud                                  | 0.30 |
| R3  | Si rugosidad supera máximo → fallo de rugosidad                                      | 0.30 |
| R4  | Si material no coincide → fallo de material                                          | 0.60 |
| R5  | Si hay grietas visibles → fallo por grietas                                          | 0.70 |
| R6  | Si hay deformación visible → fallo geométrico                                       | 0.50 |
| R7  | Si fallo diámetro **Y** fallo longitud → dimensión fuera de especificación            | 0.80 |
| R8  | Si fallo rugosidad **O** grietas → integridad superficial comprometida               | 0.80 |
| R9  | Si fallo material **Y** uso crítico → riesgo alto de operación                       | 0.90 |
| R10 | Si dimensión fuera de espec. **O** integridad comprometida **O** riesgo alto → rechazar | 0.90 |
| R11 | Si solo evidencia leve acumulada → revisión manual                                   | 0.50 |
| R12 | Si no hay fallos detectados → aceptar pieza                                          | 1.00 |

### Encadenamientos

- **Encadenamiento 1:** `diametro_fuera_tolerancia` + `longitud_fuera_tolerancia` → `dimension_fuera_especificacion` → `pieza_rechazada`
- **Encadenamiento 2:** `rugosidad_excesiva` o `grietas_detectadas` → `integridad_comprometida` → `pieza_rechazada`
- **Encadenamiento 3:** `material_incorrecto` + `uso_critico` → `riesgo_alto_operacion` → `pieza_rechazada`

In [ ]:
REGLAS = [
    {
        "id": "R1",
        "si": ["diametro_fuera_tolerancia"],
        "operador": "AND",
        "entonces": "fallo_diametro",
        "fc": 0.4,
        "explicacion": "El diámetro medido está fuera de la tolerancia especificada."
    },
    {
        "id": "R2",
        "si": ["longitud_fuera_tolerancia"],
        "operador": "AND",
        "entonces": "fallo_longitud",
        "fc": 0.3,
        "explicacion": "La longitud medida está fuera de la tolerancia especificada."
    },
    {
        "id": "R3",
        "si": ["rugosidad_excesiva"],
        "operador": "AND",
        "entonces": "fallo_rugosidad",
        "fc": 0.3,
        "explicacion": "La rugosidad superficial excede el máximo permitido."
    },
    {
        "id": "R4",
        "si": ["material_incorrecto"],
        "operador": "AND",
        "entonces": "fallo_material",
        "fc": 0.6,
        "explicacion": "El material de la pieza no coincide con el especificado."
    },
    {
        "id": "R5",
        "si": ["grietas_detectadas"],
        "operador": "AND",
        "entonces": "fallo_grietas",
        "fc": 0.7,
        "explicacion": "Se detectaron grietas visibles en la superficie de la pieza."
    },
    {
        "id": "R6",
        "si": ["deformacion_detectada"],
        "operador": "AND",
        "entonces": "fallo_geometrico",
        "fc": 0.5,
        "explicacion": "Se detectó deformación visible en la geometría de la pieza."
    },
    {
        "id": "R7",
        "si": ["fallo_diametro", "fallo_longitud"],
        "operador": "AND",
        "entonces": "dimension_fuera_especificacion",
        "fc": 0.8,
        "explicacion": "Tanto el diámetro como la longitud fallan, indicando dimensión fuera de especificación."
    },
    {
        "id": "R8",
        "si": ["fallo_rugosidad", "fallo_grietas"],
        "operador": "OR",
        "entonces": "integridad_comprometida",
        "fc": 0.8,
        "explicacion": "Rugosidad excesiva o grietas comprometen la integridad superficial."
    },
    {
        "id": "R9",
        "si": ["fallo_material", "uso_critico"],
        "operador": "AND",
        "entonces": "riesgo_alto_operacion",
        "fc": 0.9,
        "explicacion": "Material incorrecto en aplicación de uso crítico genera riesgo alto."
    },
    {
        "id": "R10",
        "si": ["dimension_fuera_especificacion", "integridad_comprometida", "riesgo_alto_operacion"],
        "operador": "OR",
        "entonces": "pieza_rechazada",
        "fc": 0.9,
        "explicacion": "Problemas dimensionales, de integridad o de riesgo alto llevan al rechazo."
    },
    {
        "id": "R11",
        "si": ["evidencia_leve"],
        "operador": "AND",
        "entonces": "requiere_revision",
        "fc": 0.5,
        "explicacion": "La evidencia acumulada es leve; se recomienda revisión manual."
    },
    {
        "id": "R12",
        "si": ["sin_fallos"],
        "operador": "AND",
        "entonces": "pieza_aceptada",
        "fc": 1.0,
        "explicacion": "No se detectaron fallos; la pieza cumple todas las especificaciones."
    }
]

print(f"Base de reglas cargada: {len(REGLAS)} reglas")
for r in REGLAS:
    op = f" {r['operador']} "
    premisas = op.join(r['si'])
    print(f"  {r['id']}: SI ({premisas}) ENTONCES {r['entonces']} [FC={r['fc']}]")

# 8. Evaluación de Evidencias

Esta función toma los datos de la pieza y las tolerancias, y genera los hechos iniciales (evidencias de fallo) como hechos binarios (`1.0` = detectado). El factor de certeza de cada evidencia lo asigna la regla correspondiente (R1–R6).

In [ ]:
def evaluar_evidencias(pieza):
    """
    Evalúa las 8 variables de entrada contra las tolerancias
    y genera los hechos iniciales.
    Los hechos de evidencia se registran con FC=1.0 (detectado/no detectado).
    El FC propio de cada evidencia lo asigna la regla correspondiente.
    """
    tipo = pieza["tipo_pieza"].lower()
    if tipo not in TOLERANCIAS:
        raise ValueError(f"Tipo de pieza '{tipo}' no reconocido. Opciones: eje, brida, tornillo")

    tol = TOLERANCIAS[tipo]
    hechos = {}

    if pieza["diametro"] < tol["diametro_min"] or pieza["diametro"] > tol["diametro_max"]:
        hechos["diametro_fuera_tolerancia"] = 1.0

    if pieza["longitud"] < tol["longitud_min"] or pieza["longitud"] > tol["longitud_max"]:
        hechos["longitud_fuera_tolerancia"] = 1.0

    if pieza["rugosidad"] > tol["rugosidad_max"]:
        hechos["rugosidad_excesiva"] = 1.0

    if pieza["material"].lower() != tol["material_esperado"]:
        hechos["material_incorrecto"] = 1.0

    if pieza["grietas_visibles"]:
        hechos["grietas_detectadas"] = 1.0

    if pieza["deformacion_visible"]:
        hechos["deformacion_detectada"] = 1.0

    if pieza["uso_critico"]:
        hechos["uso_critico"] = 1.0

    return hechos

# 9. Motor de Inferencia Hacia Adelante

El motor aplica las reglas iterativamente mientras se puedan hacer nuevas inferencias. Utiliza **encadenamiento hacia adelante**: parte de los hechos conocidos y aplica reglas hasta alcanzar conclusiones.

Cada regla se aplica **una sola vez** para evitar la sobreacumulación de evidencia (como recomienda MYCIN).

### Fórmula de combinación MYCIN

Cuando varias evidencias apoyan la misma conclusión, se combinan con:

$$
FC_{\\text{total}} = FC_{\\text{previo}} + (1 - FC_{\\text{previo}}) \\cdot FC_{\\text{nuevo}}
$$

Para reglas encadenadas, el FC de la conclusión se calcula como:

$$
FC_{\\text{conclusión}} = \\min(FC_{\\text{antecedentes}}) \\times FC_{\\text{regla}}
$$

In [ ]:
def combinar_fc(fc_previo, fc_nuevo):
    """
    Combina dos factores de certeza según la fórmula de MYCIN.
    Solo para FC positivos (evidencias de rechazo) en [0, 1].
    """
    return fc_previo + (1 - fc_previo) * fc_nuevo


def evaluar_antecedentes(hechos, condiciones, operador):
    """
    Evalúa si los antecedentes de una regla se cumplen.
    - AND: todos los antecedentes deben estar presentes. FC = mín.
    - OR:  al menos un antecedente debe estar presente. FC = máx.
    Retorna (True, fc) o (False, 0).
    """
    presentes = [c for c in condiciones if c in hechos]

    if operador == "AND":
        if len(presentes) == len(condiciones):
            fc = min(hechos[c] for c in condiciones)
            return True, fc
        return False, 0

    if operador == "OR":
        if len(presentes) > 0:
            fc = max(hechos[c] for c in presentes)
            return True, fc
        return False, 0

    return False, 0


def encadenamiento_adelante(hechos, reglas):
    """
    Motor de inferencia hacia adelante con factores de certeza.
    Cada regla se aplica una sola vez para evitar sobreacumulación.
    Se itera hasta que no se producen nuevos hechos derivados.
    Registra la traza de inferencia para el módulo de explicación.
    """
    traza = []
    reglas_aplicadas = set()
    cambios = True

    while cambios:
        cambios = False
        for regla in reglas:
            rid = regla["id"]

            # R11 y R12 se evalúan al final, no durante el encadenamiento
            if rid in ("R11", "R12"):
                continue

            # Cada regla se aplica una sola vez
            if rid in reglas_aplicadas:
                continue

            cumple, fc_ant = evaluar_antecedentes(
                hechos, regla["si"], regla["operador"]
            )
            if not cumple:
                continue

            fc_conclusion = fc_ant * regla["fc"]
            conclusion = regla["entonces"]

            if conclusion in hechos:
                hechos[conclusion] = min(combinar_fc(hechos[conclusion], fc_conclusion), 1.0)
            else:
                hechos[conclusion] = min(fc_conclusion, 1.0)

            cambios = True
            reglas_aplicadas.add(rid)

            traza.append({
                "regla": rid,
                "premisas": regla["si"],
                "operador": regla["operador"],
                "conclusion": conclusion,
                "fc": round(hechos[conclusion], 4),
                "explicacion": regla["explicacion"]
            })

    return hechos, traza

# 10. Manejo de Incertidumbre con Factores de Certeza

El FC acumulado de rechazo se calcula combinando todas las evidencias de fallo detectadas con la fórmula MYCIN.

### Criterio de decisión

| FC acumulado de rechazo | Decisión               |
|------------------------:|------------------------|
| `FC < 0.40`            | Aceptar pieza          |
| `0.40 ≤ FC < 0.60`    | Requiere revisión manual |
| `FC ≥ 0.60`           | Rechazar pieza         |

In [ ]:
def calcular_fc_rechazo(hechos):
    """
    Calcula el FC acumulado de rechazo combinando todas las evidencias
    de fallo con la fórmula MYCIN.
    """
    fallos = [
        "fallo_diametro", "fallo_longitud", "fallo_rugosidad",
        "fallo_material", "fallo_grietas", "fallo_geometrico"
    ]

    fc_total = 0.0
    for fallo in fallos:
        if fallo in hechos:
            fc_total = combinar_fc(fc_total, hechos[fallo])

    return round(min(fc_total, 1.0), 4)


def decidir(fc_rechazo):
    """
    Toma la decisión final según los umbrales definidos.
    """
    if fc_rechazo < 0.40:
        return "ACEPTAR"
    elif fc_rechazo < 0.60:
        return "REVISIÓN MANUAL"
    else:
        return "RECHAZAR"

# 11. Módulo de Explicación

El sistema no solo dice qué decisión tomó, sino **por qué**. Muestra:

- Decisión final.
- FC acumulado de rechazo.
- Lista de razones específicas.
- Reglas activadas.
- Cadena de inferencia completa.

In [ ]:
def generar_explicacion(pieza, hechos, traza, fc_rechazo, decision):
    """
    Genera un reporte legible con la decisión, razones y cadena de inferencia.
    Retorna el texto completo de la explicación.
    """
    lineas = []
    lineas.append("=" * 60)
    lineas.append(f"RESULTADO DE VERIFICACIÓN — Pieza tipo: {pieza['tipo_pieza'].upper()}")
    lineas.append("=" * 60)

    lineas.append(f"\nDatos de entrada:")
    lineas.append(f"  Tipo:        {pieza['tipo_pieza']}")
    lineas.append(f"  Diámetro:    {pieza['diametro']} mm")
    lineas.append(f"  Longitud:    {pieza['longitud']} mm")
    lineas.append(f"  Rugosidad:   {pieza['rugosidad']} Ra")
    lineas.append(f"  Material:    {pieza['material']}")
    lineas.append(f"  Grietas:     {'Sí' if pieza['grietas_visibles'] else 'No'}")
    lineas.append(f"  Deformación: {'Sí' if pieza['deformacion_visible'] else 'No'}")
    lineas.append(f"  Uso crítico: {'Sí' if pieza['uso_critico'] else 'No'}")

    lineas.append(f"\n{'─' * 60}")
    lineas.append(f"Decisión: {decision} pieza.")
    lineas.append(f"FC acumulado de rechazo: {fc_rechazo:.4f}")

    lineas.append(f"\n{'─' * 60}")
    lineas.append("Razones:")
    if not traza:
        lineas.append("  - No se detectaron fallos.")
    else:
        for paso in traza:
            lineas.append(f"  - {paso['explicacion']} (FC: {paso['fc']})")

    lineas.append(f"\n{'─' * 60}")
    lineas.append("Cadena de inferencia:")
    if not traza:
        lineas.append("  R12 → pieza_aceptada (sin fallos detectados)")
    else:
        for paso in traza:
            premisas = f" {paso['operador']} ".join(paso["premisas"])
            lineas.append(
                f"  {paso['regla']}: ({premisas}) → {paso['conclusion']} [FC={paso['fc']}]"
            )

    lineas.append("=" * 60)

    texto = "\n".join(lineas)
    print(texto)
    return texto

# 12. Sistema Experto Integrado

Función principal que orquesta todo el proceso: evaluar evidencias, ejecutar el motor de inferencia, calcular el FC acumulado, decidir y explicar.

In [ ]:
def verificar_pieza(pieza):
    """
    Ejecuta el sistema experto completo para una pieza.
    Retorna (decision, fc_rechazo, texto_explicacion).
    """
    hechos = evaluar_evidencias(pieza)

    hechos, traza = encadenamiento_adelante(hechos, REGLAS)

    fc_rechazo = calcular_fc_rechazo(hechos)

    # Evaluar reglas finales R11 y R12
    if fc_rechazo == 0:
        hechos["sin_fallos"] = 1.0
        traza_final = []
    elif fc_rechazo < 0.40:
        traza_final = traza
    else:
        if 0.40 <= fc_rechazo < 0.60:
            hechos["evidencia_leve"] = fc_rechazo
        traza_final = traza

    decision = decidir(fc_rechazo)

    texto = generar_explicacion(pieza, hechos, traza_final, fc_rechazo, decision)

    return decision, fc_rechazo, texto

# 13. Procesamiento de Lenguaje Natural Básico

El extractor convierte texto libre (proveniente de voz o entrada manual) en un diccionario estructurado con las 8 variables de entrada.

Se usa normalización de texto, búsqueda por palabras clave y expresiones regulares.

### Ejemplo de entrada:

```
La pieza es un eje con diámetro 50.4, longitud 100.1, rugosidad 1.9,
material acero, con grietas visibles, sin deformación y uso crítico.
```

### Salida esperada:

```python
{
    "tipo_pieza": "eje",
    "diametro": 50.4,
    "longitud": 100.1,
    "rugosidad": 1.9,
    "material": "acero",
    "grietas_visibles": True,
    "deformacion_visible": False,
    "uso_critico": True
}
```

In [ ]:
def normalizar_texto(texto):
    """Normaliza el texto: minúsculas y reemplaza acentos comunes."""
    texto = texto.lower().strip()
    reemplazos = {
        "á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u",
        "ü": "u", "ñ": "n"
    }
    for original, nuevo in reemplazos.items():
        texto = texto.replace(original, nuevo)
    return texto


def extraer_datos_pieza(texto):
    """
    Extrae las 8 variables de entrada desde un texto libre
    usando palabras clave y expresiones regulares.
    """
    texto_norm = normalizar_texto(texto)
    datos = {}

    # Tipo de pieza
    for tipo in ["eje", "brida", "tornillo"]:
        if tipo in texto_norm:
            datos["tipo_pieza"] = tipo
            break

    # Valores numéricos
    patron_diametro = r"diametro\s*[:=]?\s*([\d]+\.?[\d]*)"
    patron_longitud = r"longitud\s*[:=]?\s*([\d]+\.?[\d]*)"
    patron_rugosidad = r"rugosidad\s*[:=]?\s*([\d]+\.?[\d]*)"

    m = re.search(patron_diametro, texto_norm)
    if m:
        datos["diametro"] = float(m.group(1))

    m = re.search(patron_longitud, texto_norm)
    if m:
        datos["longitud"] = float(m.group(1))

    m = re.search(patron_rugosidad, texto_norm)
    if m:
        datos["rugosidad"] = float(m.group(1))

    # Material
    for mat in ["acero", "aluminio", "hierro", "bronce", "cobre"]:
        if mat in texto_norm:
            datos["material"] = mat
            break

    # Booleanos: grietas
    if re.search(r"sin\s+grietas?", texto_norm) or re.search(r"no\s+(hay\s+)?grietas?", texto_norm):
        datos["grietas_visibles"] = False
    elif re.search(r"(con\s+)?grietas?", texto_norm):
        datos["grietas_visibles"] = True
    else:
        datos["grietas_visibles"] = False

    # Booleanos: deformación
    if re.search(r"sin\s+deformacion", texto_norm) or re.search(r"no\s+(hay\s+)?deformacion", texto_norm):
        datos["deformacion_visible"] = False
    elif re.search(r"(con\s+)?deformacion", texto_norm):
        datos["deformacion_visible"] = True
    else:
        datos["deformacion_visible"] = False

    # Booleanos: uso crítico
    if re.search(r"no\s+(es\s+)?uso\s+critico", texto_norm) or re.search(r"sin\s+uso\s+critico", texto_norm):
        datos["uso_critico"] = False
    elif re.search(r"uso\s+critico", texto_norm):
        datos["uso_critico"] = True
    else:
        datos["uso_critico"] = False

    return datos

# 14. Reconocimiento de Voz y Síntesis de Voz

Se usa `speech_recognition` para capturar audio del micrófono y `pyttsx3` para leer el resultado en voz alta.

Como el reconocimiento de voz puede fallar por micrófono, ruido o conexión, el sistema incluye un **modo alternativo por texto**.

**Nota:** Si las bibliotecas de voz no están instaladas, el sistema funciona igualmente en modo texto.

In [ ]:
MODO_VOZ = False

try:
    import speech_recognition as sr
    SR_DISPONIBLE = True
except ImportError:
    SR_DISPONIBLE = False
    print("speech_recognition no disponible. Instalar con: pip install SpeechRecognition")

try:
    import pyttsx3
    TTS_DISPONIBLE = True
except ImportError:
    TTS_DISPONIBLE = False
    print("pyttsx3 no disponible. Instalar con: pip install pyttsx3")


def reconocer_voz():
    """Captura audio del micrófono y lo convierte a texto."""
    if not SR_DISPONIBLE:
        print("Reconocimiento de voz no disponible.")
        return None

    r = sr.Recognizer()
    with sr.Microphone() as source:
        print("Hable ahora... Describa la pieza a verificar.")
        r.adjust_for_ambient_noise(source)
        audio = r.listen(source)

    try:
        texto = r.recognize_google(audio, language="es-ES")
        print(f"Texto reconocido: {texto}")
        return texto
    except sr.UnknownValueError:
        print("No se pudo entender el audio.")
        return None
    except sr.RequestError:
        print("Error de conexión con el servicio de reconocimiento.")
        return None


def sintetizar_voz(texto):
    """Lee en voz alta el texto del resultado."""
    if not TTS_DISPONIBLE:
        print("(Síntesis de voz no disponible — mostrando texto)")
        return

    engine = pyttsx3.init()
    engine.say(texto)
    engine.runAndWait()


def obtener_entrada():
    """
    Obtiene la descripción de la pieza por voz o por texto,
    según el modo configurado.
    """
    if MODO_VOZ and SR_DISPONIBLE:
        texto = reconocer_voz()
        if texto:
            return texto
        print("Falló la voz; cambiando a modo texto.")

    return input("Describa la pieza (ejemplo: Es un eje, diámetro 50.4, longitud 100, rugosidad 1.9, material acero, sin grietas, sin deformación, uso crítico): ")


print(f"Modo voz: {'Activado' if MODO_VOZ else 'Desactivado (modo texto)'}")
print(f"speech_recognition: {'Disponible' if SR_DISPONIBLE else 'No disponible'}")
print(f"pyttsx3: {'Disponible' if TTS_DISPONIBLE else 'No disponible'}")

# 15. Chatbot Interactivo

Función que integra la entrada (voz o texto), el procesamiento NLP, el sistema experto y la salida (voz o texto).

In [ ]:
def chatbot():
    """
    Chatbot interactivo que:
    1. Captura entrada por voz o texto.
    2. Extrae variables con NLP.
    3. Ejecuta el sistema experto.
    4. Lee el resultado en voz alta.
    """
    print("\n" + "=" * 60)
    print("CHATBOT EXPERTO — Verificación de Piezas Mecánicas")
    print("=" * 60)

    texto = obtener_entrada()
    print(f"\nEntrada recibida: {texto}")

    datos = extraer_datos_pieza(texto)
    print(f"\nDatos extraídos: {datos}")

    campos = ["tipo_pieza", "diametro", "longitud", "rugosidad",
              "material", "grietas_visibles", "deformacion_visible", "uso_critico"]
    faltantes = [c for c in campos if c not in datos]

    if faltantes:
        print(f"\nNo se pudieron extraer: {faltantes}")
        print("Por favor, ingrese los datos manualmente.")
        return None

    print("\n")
    decision, fc, explicacion = verificar_pieza(datos)

    # Síntesis de voz del resultado
    resumen = f"Decisión: {decision}. Factor de certeza de rechazo: {fc:.2f}"
    sintetizar_voz(resumen)

    return decision, fc

---

# 16. Pruebas de Validación

Se ejecutan 5 pruebas con resultados esperados para validar el correcto funcionamiento del sistema.

## Prueba 1: Pieza aceptada

Un eje con todos los valores dentro de tolerancia. Sin fallos.

**Resultado esperado:** FC ≈ 0, decisión: ACEPTAR.

In [ ]:
pieza_1 = {
    "tipo_pieza": "eje",
    "diametro": 50.0,
    "longitud": 100.0,
    "rugosidad": 1.5,
    "material": "acero",
    "grietas_visibles": False,
    "deformacion_visible": False,
    "uso_critico": False
}

decision_1, fc_1, _ = verificar_pieza(pieza_1)

## Prueba 2: Rechazo por material incorrecto y grietas

Una brida cuyo material es acero (debería ser aluminio) y tiene grietas visibles.

**Resultado esperado:** FC acumulado alto, decisión: RECHAZAR.

In [ ]:
pieza_2 = {
    "tipo_pieza": "brida",
    "diametro": 80.2,
    "longitud": 20.0,
    "rugosidad": 1.8,
    "material": "acero",
    "grietas_visibles": True,
    "deformacion_visible": False,
    "uso_critico": True
}

decision_2, fc_2, _ = verificar_pieza(pieza_2)

## Prueba 3: Rechazo por encadenamiento dimensional

Un tornillo con diámetro y longitud fuera de tolerancia, que activan el **encadenamiento 1**:

`diametro_fuera_tolerancia` + `longitud_fuera_tolerancia` → `dimension_fuera_especificacion` → `pieza_rechazada`

**Resultado esperado:** Inferencia encadenada, decisión: REVISIÓN o RECHAZAR según FC.

In [ ]:
pieza_3 = {
    "tipo_pieza": "tornillo",
    "diametro": 10.5,
    "longitud": 21.5,
    "rugosidad": 1.2,
    "material": "acero",
    "grietas_visibles": False,
    "deformacion_visible": False,
    "uso_critico": False
}

decision_3, fc_3, _ = verificar_pieza(pieza_3)

## Prueba 4: Revisión manual

Un eje con el diámetro ligeramente fuera de tolerancia (50.3 mm, tolerancia 49.8–50.2).

**Resultado esperado:** FC ≈ 0.40, decisión: REVISIÓN MANUAL.

In [ ]:
pieza_4 = {
    "tipo_pieza": "eje",
    "diametro": 50.3,
    "longitud": 100.0,
    "rugosidad": 1.7,
    "material": "acero",
    "grietas_visibles": False,
    "deformacion_visible": False,
    "uso_critico": False
}

decision_4, fc_4, _ = verificar_pieza(pieza_4)

## Prueba 5: Procesamiento de texto (NLP)

Entrada en lenguaje natural para validar el extractor de datos.

**Resultado esperado:** Extraer correctamente las 8 variables, detectar diámetro fuera de tolerancia, detectar rugosidad excesiva.

In [ ]:
texto_prueba = (
    "Es un eje, diámetro 50.4, longitud 100.2, rugosidad 2.3, "
    "material acero, sin grietas, sin deformación, uso crítico."
)

print("Texto de entrada:")
print(f"  {texto_prueba}")
print()

datos_extraidos = extraer_datos_pieza(texto_prueba)
print("Datos extraídos por NLP:")
for k, v in datos_extraidos.items():
    print(f"  {k}: {v}")

campos = ["tipo_pieza", "diametro", "longitud", "rugosidad",
          "material", "grietas_visibles", "deformacion_visible", "uso_critico"]
faltantes = [c for c in campos if c not in datos_extraidos]
print(f"\nVariables faltantes: {faltantes if faltantes else 'Ninguna — extracción completa'}")

print("\n" + "=" * 60)
print("Ejecutando sistema experto con datos extraídos:")
print("=" * 60 + "\n")

decision_5, fc_5, _ = verificar_pieza(datos_extraidos)

## Resumen de Pruebas

In [ ]:
print("\n" + "=" * 60)
print("RESUMEN DE PRUEBAS DE VALIDACIÓN")
print("=" * 60)
print(f"{'Prueba':<12} {'Tipo':<12} {'FC Rechazo':>12} {'Decisión':<20}")
print("-" * 56)
print(f"{'Prueba 1':<12} {'Eje':<12} {fc_1:>12.4f} {decision_1:<20}")
print(f"{'Prueba 2':<12} {'Brida':<12} {fc_2:>12.4f} {decision_2:<20}")
print(f"{'Prueba 3':<12} {'Tornillo':<12} {fc_3:>12.4f} {decision_3:<20}")
print(f"{'Prueba 4':<12} {'Eje':<12} {fc_4:>12.4f} {decision_4:<20}")
print(f"{'Prueba 5':<12} {'Eje (NLP)':<12} {fc_5:>12.4f} {decision_5:<20}")
print("=" * 56)

---

# 17. Conclusiones y Preguntas de Análisis

## 1. Importancia del tema seleccionado

El **control de calidad** en piezas mecanizadas es crítico en la industria. Un sistema experto que automatice la verificación dimensional y de acabado permite:

- Reducir errores humanos en la inspección.
- Disminuir rechazos no justificados.
- Aumentar la seguridad industrial al detectar piezas defectuosas antes de su uso.
- Estandarizar los criterios de aceptación o rechazo.

## 2. Limitaciones del reconocimiento de voz

- **Ruido ambiental:** en entornos industriales, el ruido puede interferir con el reconocimiento.
- **Acentos y variaciones dialectales:** el modelo puede no transcribir correctamente ciertos acentos del español.
- **Errores de transcripción:** números y unidades pueden transcribirse incorrectamente (por ejemplo, "cincuenta punto cuatro" en vez de "50.4").
- **Dependencia del micrófono:** la calidad del hardware afecta directamente la precisión.
- **Conexión a internet:** `recognize_google` requiere conectividad.

Por estas razones, se incluye un **modo alternativo por texto**.

## 3. Formalización del lenguaje natural

El paso de texto libre a hechos lógicos (predicados) requiere:

- Normalización del texto (minúsculas, eliminación de acentos).
- Identificación de palabras clave (tipo de pieza, materiales).
- Extracción de valores numéricos con expresiones regulares.
- Resolución de negaciones ("sin grietas" vs. "con grietas").

Este es un procesamiento de lenguaje natural básico, pero suficiente para el dominio acotado de este taller.

## 4. Ventajas de la lógica simbólica

- **Trazabilidad:** cada decisión se puede explicar con la cadena de reglas activadas.
- **Explicabilidad:** el módulo de explicación muestra exactamente por qué se rechaza o acepta.
- **Auditabilidad:** las reglas son legibles y pueden ser validadas por un experto humano.
- **Mantenibilidad:** agregar, modificar o eliminar reglas no requiere reentrenar un modelo.

## 5. Escalamiento industrial

Para llevar este sistema a un entorno industrial real, se podría:

- Conectar **sensores** de medición (CMM, rugosímetros) para capturar datos automáticamente.
- Integrar con **bases de datos** de históricos de producción.
- Agregar más **tipos de piezas** y tolerancias.
- Implementar **calibración** y monitoreo en línea.
- Combinar con técnicas de **aprendizaje automático** para ajustar umbrales según datos históricos.

---

# Explicación del Motor de Inferencia

## Encadenamiento hacia adelante

El motor implementa **encadenamiento hacia adelante** (*forward chaining*):

1. Se parte de los **hechos conocidos** (evidencias de fallo evaluadas contra tolerancias).
2. Se itera sobre todas las reglas. Si los antecedentes de una regla se cumplen, se agrega la conclusión como nuevo hecho.
3. Cada regla se aplica **una sola vez** para evitar la sobreacumulación de evidencia.
4. Se repite hasta que no se generan nuevos hechos (punto fijo).

## Fórmula MYCIN para combinación de evidencias

Cuando dos evidencias distintas apoyan la misma conclusión, la certeza acumulada se calcula con:

$$FC_{\\text{total}} = FC_{\\text{previo}} + (1 - FC_{\\text{previo}}) \\cdot FC_{\\text{nuevo}}$$

Esto asegura que:
- La certeza crece monótonamente.
- Nunca supera 1.0.
- Cada nueva evidencia aporta proporcionalmente a lo no cubierto.

## FC en reglas encadenadas

Para reglas con múltiples antecedentes (AND), el FC de la conclusión es:

$$FC_{\\text{conclusión}} = \\min(FC_{\\text{antecedentes}}) \\times FC_{\\text{regla}}$$

Esto es conservador: la certeza de la conclusión no puede ser mayor que la del antecedente más débil.